In [1]:
import os
import re
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("../data/slm_eval_dataset_hard.csv")

print("Dataset shape:", df.shape)
print(df["expected_risk"].value_counts())

df.head()

Dataset shape: (30, 2)
expected_risk
Medium    12
High      10
Low        8
Name: count, dtype: int64


,input,expected_risk
0,License: MIT License Family: Permissive Projec...,High
1,License: Apache-2.0 License Family: Permissive...,Medium
2,License: BSD-3-Clause License Family: Permissi...,Medium
3,License: LGPL-2.1 License Family: Weak Copylef...,Low
4,License: LGPL-3.0 License Family: Weak Copylef...,High


In [3]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

print("Tokenizer loaded.")

Tokenizer loaded.


In [4]:
def build_prompt(scenario):
    return f"""
You are an open-source license compliance assistant.

Analyze the following software dependency scenario and classify the compliance risk.

Risk categories:
- Low: low legal/compliance concern or obligations are minimal/satisfied
- Medium: moderate compliance concern requiring review or conditions
- High: serious compliance concern, strong copyleft exposure, unknown terms, missing obligations, or restricted redistribution

Scenario:
{scenario}

Return exactly in this format:

Risk: <Low|Medium|High>
Reason: <one short sentence>
""".strip()

In [5]:
def extract_risk(response):
    for line in response.split("\n"):
        line_clean = line.strip().lower()

        if line_clean.startswith("risk:"):
            value = line_clean.replace("risk:", "").strip()

            if value.startswith("high"):
                return "High"
            elif value.startswith("medium"):
                return "Medium"
            elif value.startswith("low"):
                return "Low"

    return "Unknown"


def generate_response(model, scenario):
    prompt = build_prompt(scenario)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [6]:
def evaluate_model(model, model_label, dataset):
    responses = []
    predictions = []

    for idx, row in dataset.iterrows():
        print(f"{model_label} | Processing {idx + 1}/{len(dataset)}")

        response = generate_response(model, row["input"])
        predicted_risk = extract_risk(response)

        responses.append(response)
        predictions.append(predicted_risk)

    result_df = dataset.copy()
    result_df[f"{model_label}_response"] = responses
    result_df[f"{model_label}_predicted_risk"] = predictions

    y_true = result_df["expected_risk"]
    y_pred = result_df[f"{model_label}_predicted_risk"]

    accuracy = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print("\n" + "=" * 70)
    print(model_label)
    print("=" * 70)
    print("Accuracy:", accuracy)
    print("\nClassification Report:")
    print(report)
    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model_label": model_label,
        "accuracy": accuracy,
        "report": report,
        "confusion_matrix": cm,
        "result_df": result_df
    }

In [7]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

base_results = evaluate_model(
    base_model,
    "base_qwen",
    df
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

base_qwen | Processing 1/30
base_qwen | Processing 2/30
base_qwen | Processing 3/30
base_qwen | Processing 4/30
base_qwen | Processing 5/30
base_qwen | Processing 6/30
base_qwen | Processing 7/30
base_qwen | Processing 8/30
base_qwen | Processing 9/30
base_qwen | Processing 10/30
base_qwen | Processing 11/30
base_qwen | Processing 12/30
base_qwen | Processing 13/30
base_qwen | Processing 14/30
base_qwen | Processing 15/30
base_qwen | Processing 16/30
base_qwen | Processing 17/30
base_qwen | Processing 18/30
base_qwen | Processing 19/30
base_qwen | Processing 20/30
base_qwen | Processing 21/30
base_qwen | Processing 22/30
base_qwen | Processing 23/30
base_qwen | Processing 24/30
base_qwen | Processing 25/30
base_qwen | Processing 26/30
base_qwen | Processing 27/30
base_qwen | Processing 28/30
base_qwen | Processing 29/30
base_qwen | Processing 30/30

base_qwen
Accuracy: 0.3

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.10  

In [8]:
del base_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Base model cleared from memory.")

Base model cleared from memory.


In [9]:
base_model_v1 = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model_v1 = PeftModel.from_pretrained(
    base_model_v1,
    "../models/qwen_oss_compliance_lora"
)

v1_results = evaluate_model(
    model_v1,
    "finetuned_qwen_v1",
    df
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

finetuned_qwen_v1 | Processing 1/30
finetuned_qwen_v1 | Processing 2/30
finetuned_qwen_v1 | Processing 3/30
finetuned_qwen_v1 | Processing 4/30
finetuned_qwen_v1 | Processing 5/30
finetuned_qwen_v1 | Processing 6/30
finetuned_qwen_v1 | Processing 7/30
finetuned_qwen_v1 | Processing 8/30
finetuned_qwen_v1 | Processing 9/30
finetuned_qwen_v1 | Processing 10/30
finetuned_qwen_v1 | Processing 11/30
finetuned_qwen_v1 | Processing 12/30
finetuned_qwen_v1 | Processing 13/30
finetuned_qwen_v1 | Processing 14/30
finetuned_qwen_v1 | Processing 15/30
finetuned_qwen_v1 | Processing 16/30
finetuned_qwen_v1 | Processing 17/30
finetuned_qwen_v1 | Processing 18/30
finetuned_qwen_v1 | Processing 19/30
finetuned_qwen_v1 | Processing 20/30
finetuned_qwen_v1 | Processing 21/30
finetuned_qwen_v1 | Processing 22/30
finetuned_qwen_v1 | Processing 23/30
finetuned_qwen_v1 | Processing 24/30
finetuned_qwen_v1 | Processing 25/30
finetuned_qwen_v1 | Processing 26/30
finetuned_qwen_v1 | Processing 27/30
finetuned_

In [10]:
del model_v1
del base_model_v1

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Fine-tuned v1 cleared from memory.")

Fine-tuned v1 cleared from memory.


In [11]:
base_model_v2 = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model_v2 = PeftModel.from_pretrained(
    base_model_v2,
    "../models/qwen_oss_compliance_lora_v2"
)

v2_results = evaluate_model(
    model_v2,
    "finetuned_qwen_v2",
    df
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

finetuned_qwen_v2 | Processing 1/30
finetuned_qwen_v2 | Processing 2/30
finetuned_qwen_v2 | Processing 3/30
finetuned_qwen_v2 | Processing 4/30
finetuned_qwen_v2 | Processing 5/30
finetuned_qwen_v2 | Processing 6/30
finetuned_qwen_v2 | Processing 7/30
finetuned_qwen_v2 | Processing 8/30
finetuned_qwen_v2 | Processing 9/30
finetuned_qwen_v2 | Processing 10/30
finetuned_qwen_v2 | Processing 11/30
finetuned_qwen_v2 | Processing 12/30
finetuned_qwen_v2 | Processing 13/30
finetuned_qwen_v2 | Processing 14/30
finetuned_qwen_v2 | Processing 15/30
finetuned_qwen_v2 | Processing 16/30
finetuned_qwen_v2 | Processing 17/30
finetuned_qwen_v2 | Processing 18/30
finetuned_qwen_v2 | Processing 19/30
finetuned_qwen_v2 | Processing 20/30
finetuned_qwen_v2 | Processing 21/30
finetuned_qwen_v2 | Processing 22/30
finetuned_qwen_v2 | Processing 23/30
finetuned_qwen_v2 | Processing 24/30
finetuned_qwen_v2 | Processing 25/30
finetuned_qwen_v2 | Processing 26/30
finetuned_qwen_v2 | Processing 27/30
finetuned_

In [12]:
comparison_df = pd.DataFrame({
    "Model": [
        "Base Qwen",
        "Fine-Tuned Qwen v1",
        "Fine-Tuned Qwen v2"
    ],
    "Accuracy": [
        base_results["accuracy"],
        v1_results["accuracy"],
        v2_results["accuracy"]
    ]
})

comparison_df

,Model,Accuracy
0,Base Qwen,0.300000
1,Fine-Tuned Qwen v1,0.466667
2,Fine-Tuned Qwen v2,0.433333


In [13]:
os.makedirs("../outputs", exist_ok=True)

base_results["result_df"].to_csv(
    "../outputs/step7C_base_qwen_hard_predictions.csv",
    index=False
)

v1_results["result_df"].to_csv(
    "../outputs/step7C_finetuned_qwen_v1_hard_predictions.csv",
    index=False
)

v2_results["result_df"].to_csv(
    "../outputs/step7C_finetuned_qwen_v2_hard_predictions.csv",
    index=False
)

comparison_df.to_csv(
    "../outputs/step7C_hard_model_comparison.csv",
    index=False
)

with open("../outputs/step7C_hard_evaluation_report.txt", "w") as f:
    f.write("Step 7C — Hard Evaluation Model Comparison\n\n")

    for result in [base_results, v1_results, v2_results]:
        f.write("=" * 70 + "\n")
        f.write(result["model_label"] + "\n")
        f.write("=" * 70 + "\n")
        f.write(f"Accuracy: {result['accuracy']}\n\n")
        f.write("Classification Report:\n")
        f.write(result["report"])
        f.write("\nConfusion Matrix:\n")
        f.write(str(result["confusion_matrix"]))
        f.write("\n\n")

print("Step 7C outputs saved successfully.")

Step 7C outputs saved successfully.


In [14]:
os.makedirs("../outputs", exist_ok=True)

base_results["result_df"].to_csv(
    "../outputs/step7C_base_qwen_hard_predictions.csv",
    index=False
)

v1_results["result_df"].to_csv(
    "../outputs/step7C_finetuned_qwen_v1_hard_predictions.csv",
    index=False
)

v2_results["result_df"].to_csv(
    "../outputs/step7C_finetuned_qwen_v2_hard_predictions.csv",
    index=False
)

comparison_df.to_csv(
    "../outputs/step7C_hard_model_comparison.csv",
    index=False
)

with open("../outputs/step7C_hard_evaluation_report.txt", "w") as f:
    f.write("Step 7C — Hard Evaluation Model Comparison\n\n")

    for result in [base_results, v1_results, v2_results]:
        f.write("=" * 70 + "\n")
        f.write(result["model_label"] + "\n")
        f.write("=" * 70 + "\n")
        f.write(f"Accuracy: {result['accuracy']}\n\n")
        f.write("Classification Report:\n")
        f.write(result["report"])
        f.write("\nConfusion Matrix:\n")
        f.write(str(result["confusion_matrix"]))
        f.write("\n\n")

print("Step 7C outputs saved successfully.")

Step 7C outputs saved successfully.
